<a href="https://colab.research.google.com/github/Kiris-02/vox-persona/blob/main/vox_gpt_sovits_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ VOX IMPERIUM — GPT-SoVITS Cloud GPU Voice Clone Engine

**Mượn miễn phí card đồ họa NVIDIA T4 (16GB VRAM) của Google** để chạy GPT-SoVITS từ GitHub, clone giọng thật 100% cho 6 nhân vật:
- **Steve Jobs, Donald Trump, Elon Musk, Mark Zuckerberg, Nikola Tesla, Xi Jinping.**

👉 **Hướng dẫn chạy (Chỉ 1 thao tác duy nhất):**
1. Trên thanh menu, bấm **Runtime** (Thời gian chạy) -> chọn **Run all** (Chạy tất cả) hoặc bấm phím tắt `Ctrl + F9`.
2. Chờ hệ thống cài đặt và tải mô hình (khoảng 3-5 phút).
3. Ở ô code cuối cùng, copy đường link `https://xxx.trycloudflare.com` và dán vào mục Cài đặt trên web `vox-persona` là xong!

In [1]:
# 1. Kiểm tra card đồ họa NVIDIA T4
!nvidia-smi

Thu Sep 10 04:21:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Cấu hình kịch bản cài đặt GPT-SoVITS
%%writefile /content/setup.sh
set -e
cd /content

if [ ! -d "GPT-SoVITS" ]; then
    git clone https://github.com/RVC-Boss/GPT-SoVITS.git
fi

cd GPT-SoVITS

if conda env list | awk '{print $1}' | grep -Fxq "GPTSoVITS"; then
    :
else
    conda create -n GPTSoVITS python=3.10 -y
fi

source activate GPTSoVITS
bash install.sh --device CU126 --source HF --download-uvr5
pip install -q ipykernel pycloudflared fastapi uvicorn requests


Writing /content/setup.sh


In [ ]:
# 3. Tải Miniconda và tiến hành cài đặt GPT-SoVITS
%pip install -q condacolab
import condacolab
condacolab.install_from_url("https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh")
!cd /content && bash setup.sh


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:01:58
🔁 Restarting kernel...
Cloning into 'GPT-SoVITS'...
remote: Enumerating objects: 5974, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 5974 (delta 3), reused 0 (delta 0), pack-reused 5954 (from 3)
Receiving objects: 100% (5974/5974), 14.27 MiB | 12.75 MiB/s, done.

In [ ]:
# 4. Tải các file mẫu âm thanh giọng thật của 6 nhân vật và chuẩn hóa sang WAV 32kHz
import os, urllib.request, subprocess, shutil

os.makedirs('/content/GPT-SoVITS/refs', exist_ok=True)
base_url = 'https://raw.githubusercontent.com/Kiris-02/vox-persona/main/assets/audio/'
files = [
    ('jobs', 'jobs_speech.mp3'),
    ('trump', 'trump_speech.mp3'),
    ('musk', 'musk_speech.wav'),
    ('zuck', 'zuck_speech.mp3'),
    ('tesla', 'tesla_speech.mp3'),
    ('xijinping', 'xijinping_speech.ogg')
]

# Ưu tiên tìm ffmpeg trong conda env GPTSoVITS trước, sau đó đến system
ffmpeg_bin = None
conda_ffmpeg = "/usr/local/envs/GPTSoVITS/bin/ffmpeg"
if os.path.exists(conda_ffmpeg) and os.access(conda_ffmpeg, os.X_OK):
    ffmpeg_bin = conda_ffmpeg
else:
    ffmpeg_bin = shutil.which("ffmpeg")

if not ffmpeg_bin:
    raise RuntimeError("❌ [BLOCKER] Không tìm thấy binary ffmpeg! Vui lòng kiểm tra lại bước cài đặt Conda env.")

print(f"✓ Sử dụng FFmpeg binary: {ffmpeg_bin}")

for p, f in files:
    raw_dest = f'/content/GPT-SoVITS/refs/{f}'
    wav_dest = f'/content/GPT-SoVITS/refs/{p}_ref.wav'
    if not os.path.exists(raw_dest):
        print(f'Downloading reference audio: {f}...')
        urllib.request.urlretrieve(base_url + f, raw_dest)
    
    if not os.path.exists(raw_dest) or os.path.getsize(raw_dest) < 1000:
        raise RuntimeError(f"❌ [BLOCKER] File reference audio {f} tải về bị rỗng hoặc không tồn tại!")

    # Chuẩn hóa âm thanh sang định dạng chuẩn 32kHz 16-bit Mono WAV cho GPT-SoVITS V2
    if not os.path.exists(wav_dest) or os.path.getsize(wav_dest) < 1000:
        cmd = [ffmpeg_bin, '-y', '-i', raw_dest, '-ar', '32000', '-ac', '1', '-c:a', 'pcm_s16le', wav_dest]
        res = subprocess.run(cmd, capture_output=True, text=True)
        if res.returncode != 0:
            print(f"❌ [FFMPEG ERROR] Chi tiết lỗi khi convert {f}:")
            print(res.stderr)
            raise RuntimeError(f"FFmpeg conversion failed for {f} (exit code {res.returncode}): {res.stderr}")
        
        if not os.path.exists(wav_dest) or os.path.getsize(wav_dest) < 1000:
            raise RuntimeError(f"❌ [BLOCKER] File WAV đích {wav_dest} không được tạo hoặc rỗng sau khi ffmpeg chạy!")

        print(f'✓ {p}: Đã chuẩn hóa thành {p}_ref.wav ({os.path.getsize(wav_dest)} bytes)')

print('\n✓ Toàn bộ 6 mẫu giọng nhân vật đã được nạp & chuẩn hóa WAV 32kHz sẵn sàng 100%!')

In [ ]:
# 5. Tạo Bridge Server kết nối web vox-persona với GPT-SoVITS
%%writefile /content/GPT-SoVITS/bridge_server.py
import os, requests
from fastapi import FastAPI, Query, HTTPException
from fastapi.responses import Response
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

app = FastAPI(title='Vox Imperium GPT-SoVITS Bridge')
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)

PERSONAS = {
    'jobs': {
        'ref': '/content/GPT-SoVITS/refs/jobs_ref.wav',
        'text': "Your time is limited, so don't waste it living someone else's life.",
        'lang': 'en'
    },
    'trump': {
        'ref': '/content/GPT-SoVITS/refs/trump_ref.wav',
        'text': "Look, nobody understands deals better than me, believe me.",
        'lang': 'en'
    },
    'musk': {
        'ref': '/content/GPT-SoVITS/refs/musk_ref.wav',
        'text': 'From a physics first principles perspective, everything is just energy conversion.',
        'lang': 'en'
    },
    'zuck': {
        'ref': '/content/GPT-SoVITS/refs/zuck_ref.wav',
        'text': 'Move fast and break things, open ecosystems always win.',
        'lang': 'en'
    },
    'tesla': {
        'ref': '/content/GPT-SoVITS/refs/tesla_ref.wav',
        'text': 'If you only knew the magnificence of the 3, 6, and 9, you would have a key to the universe.',
        'lang': 'en'
    },
    'xijinping': {
        'ref': '/content/GPT-SoVITS/refs/xijinping_ref.wav',
        'text': '你好。历史的长河奔腾向前。',
        'lang': 'zh'
    }
}

@app.get('/health')
def health():
    return {'status': 'live', 'engine': 'GPT-SoVITS', 'personas': list(PERSONAS.keys())}

@app.get('/synthesize')
def synthesize(persona: str = Query('musk'), text: str = Query(...), lang: str = Query(None)):
    p = persona.lower().strip()
    if p not in PERSONAS:
        p = 'musk'
    cfg = PERSONAS[p]

    text_lang = lang if lang else ('zh' if p == 'xijinping' else 'en')

    payload = {
        'text': text,
        'text_lang': text_lang,
        'ref_audio_path': cfg['ref'],
        'prompt_text': cfg['text'],
        'prompt_lang': cfg['lang'],
        'streaming_mode': False
    }

    try:
        r = requests.post('http://127.0.0.1:9880/tts', json=payload, timeout=60)
        if r.status_code == 200:
            return Response(content=r.content, media_type='audio/wav')
        else:
            raise HTTPException(status_code=r.status_code, detail=r.text)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == '__main__':
    uvicorn.run(app, host='0.0.0.0', port=8000)


In [ ]:
# 6. Khởi chạy GPT-SoVITS API V2 & Mở đường hầm Cloudflare Tunnel
import sys, subprocess

# 1. Đảm bảo cài đặt các gói cần thiết vào chính xác Python interpreter của notebook
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycloudflared", "requests"], check=True)

import time, socket, re, os
import requests

def is_port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1.0)
        return s.connect_ex(('127.0.0.1', port)) == 0

# 2. Khởi chạy lõi AI GPT-SoVITS trên GPU NVIDIA T4
api_log_path = "/content/gpt_sovits_api.log"

if is_port_open(9880):
    print("✓ Lõi AI GPT-SoVITS đã đang chạy sẵn trên cổng 9880.")
else:
    print("1. Đang khởi động lõi AI GPT-SoVITS V2 trên GPU NVIDIA T4...")
    api_log_file = open(api_log_path, "w", encoding="utf-8", errors="replace")
    api_proc = subprocess.Popen(
        ["bash", "-c", "source activate GPTSoVITS && python -u api_v2.py -a 127.0.0.1 -p 9880 -c GPT_SoVITS/configs/tts_infer.yaml"],
        cwd="/content/GPT-SoVITS",
        stdout=api_log_file,
        stderr=subprocess.STDOUT
    )
    
    # Chờ mô hình load vào VRAM và kiểm tra process liên tục
    ready = False
    start_time = time.time()
    while time.time() - start_time < 90:
        # Kiểm tra nếu tiến trình bị crash/chết đột ngột
        ret_code = api_proc.poll()
        if ret_code is not None:
            api_log_file.flush()
            with open(api_log_path, "r", encoding="utf-8", errors="replace") as f:
                logs = f.read()
            print(f"\n❌ [BLOCKER] GPT-SoVITS API đã crash với mã lỗi {ret_code}!")
            print("="*40 + " LOG LỖI GPT-SOVITS " + "="*40)
            print(logs.strip()[-2500:] if len(logs) > 2500 else logs.strip())
            print("="*100)
            raise RuntimeError(f"GPT-SoVITS API exited with code {ret_code}. Xem log chi tiết ở trên.")
        
        if is_port_open(9880):
            ready = True
            break
            
        time.sleep(2)
        print(f"   Đang nạp weights vào VRAM ({int(time.time() - start_time)}s)...", end="\r")
    
    if not ready:
        api_log_file.flush()
        with open(api_log_path, "r", encoding="utf-8", errors="replace") as f:
            logs = f.read()
        print("\n❌ [BLOCKER] Hết thời gian chờ (90s) nhưng cổng 9880 không mở!")
        print("="*40 + " LOG HIỆN TẠI " + "="*40)
        print(logs.strip()[-2500:] if len(logs) > 2500 else logs.strip())
        print("="*100)
        raise TimeoutError("Cổng 9880 không phản hồi trong 90s. Tiến trình đã bị hủy.")
        
    print("\n✓ Mô hình GPT-SoVITS V2 đã nạp thành công vào GPU!")

# 3. Khởi chạy Bridge Server kết nối web
bridge_log_path = "/content/bridge_server.log"

if is_port_open(8000):
    print("✓ Bridge Server đã đang chạy sẵn trên cổng 8000.")
else:
    print("2. Đang khởi động Bridge Server kết nối web...")
    bridge_log_file = open(bridge_log_path, "w", encoding="utf-8", errors="replace")
    bridge_proc = subprocess.Popen(
        ["bash", "-c", "source activate GPTSoVITS && python -u bridge_server.py"],
        cwd="/content/GPT-SoVITS",
        stdout=bridge_log_file,
        stderr=subprocess.STDOUT
    )
    
    bridge_ready = False
    for _ in range(15):
        if bridge_proc.poll() is not None:
            break
        if is_port_open(8000):
            bridge_ready = True
            break
        time.sleep(1)
    
    if not bridge_ready or not is_port_open(8000):
        bridge_log_file.flush()
        with open(bridge_log_path, "r", encoding="utf-8", errors="replace") as f:
            b_logs = f.read()
        print("\n❌ [BLOCKER] Bridge Server không khởi động được trên cổng 8000!")
        print(b_logs.strip())
        raise RuntimeError(f"Bridge Server failed to start on port 8000. Logs: {b_logs.strip()[-500:]}")
        
    print("✓ Bridge Server đã sẵn sàng trên cổng 8000!")

# 4. Kiểm tra sức khỏe Bridge & Chạy Self-Test TRƯỚC KHI MỞ TUNNEL
print("3. Kiểm tra kết nối cục bộ và chạy Self-Test tạo giọng nói Steve Jobs...")
health_res = requests.get("http://127.0.0.1:8000/health", timeout=5)
if health_res.status_code != 200:
    raise RuntimeError(f"❌ [BLOCKER] Bridge /health check thất bại ({health_res.status_code}): {health_res.text}")

test_res = requests.get("http://127.0.0.1:8000/synthesize?persona=jobs&text=Testing+GPT+SoVITS+voice+engine", timeout=60)
if test_res.status_code != 200 or len(test_res.content) < 500:
    raise RuntimeError(f"❌ [BLOCKER] Self-test synthesis thất bại ({test_res.status_code}): {test_res.text[:300]}")

print(f"✅ KIỂM TRA CỤC BỘ THÀNH CÔNG 100%! Đã tạo âm thanh mẫu ({len(test_res.content)} bytes) hoàn hảo.")

# 5. CHỈ KHI TEST THÀNH CÔNG -> Mới mở đường hầm Cloudflare Tunnel
print("4. Đang mở đường hầm Cloudflare Tunnel bảo mật...")
tunnel_url = None

# Thử pycloudflared trước
try:
    from pycloudflared import try_cloudflare
    public_tunnel = try_cloudflare(port=8000)
    if public_tunnel and hasattr(public_tunnel, 'tunnel'):
        cand = str(public_tunnel.tunnel).strip()
        if "trycloudflare.com" in cand:
            tunnel_url = cand
            print(f"✓ pycloudflared đã kết nối thành công: {tunnel_url}")
except Exception as e:
    print(f"ℹ️ pycloudflared thông báo ({e}), chuyển sang chế độ Cloudflared Binary dự phòng...")

# Fallback sang binary cloudflared chính thức nếu cần
if not tunnel_url:
    if not os.path.exists("/usr/local/bin/cloudflared"):
        print("   Đang tải cloudflared binary chính thức từ Cloudflare...")
        subprocess.run(
            "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
            shell=True, check=True
        )
    
    cf_proc = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
        stderr=subprocess.PIPE,
        stdout=subprocess.PIPE,
        universal_newlines=True
    )
    for _ in range(40):
        line = cf_proc.stderr.readline()
        if not line:
            time.sleep(0.5)
            continue
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            tunnel_url = match.group(0)
            break

if not tunnel_url:
    raise RuntimeError("❌ [BLOCKER] Không thể tạo Cloudflare Tunnel qua cả pycloudflared và cloudflared binary!")

print("\n" + "="*70)
print("🎉 ĐƯỜNG LINK API GPT-SoVITS CỦA BẠN ĐÃ HOÀN TẤT VÀ KIỂM TRA XONG 100%:")
print(f"👉 {tunnel_url}")
print("="*70)
print("Copy đường link trên và dán vào ô GPT-SoVITS trên web vox-persona!\n")
